In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import flowmap
from flowmap import VectorFieldEmbedder

# -----------------------------
# Load data
# -----------------------------
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")

# -----------------------------
# Load saved embedding
# -----------------------------
X_emb = pd.read_csv("./data/larry/larry_flowmap_embedding.csv").values

# -----------------------------
# Expression matrix
# -----------------------------
hvg = adata.var["highly_variable"].values

X = adata.layers["spliced"][:, hvg]
X = X.toarray() if hasattr(X, "toarray") else X
X = np.log1p(X)
X = StandardScaler().fit_transform(X)

# -----------------------------
# Velocity matrix
# -----------------------------
V = adata.layers["velocity"][:, hvg]
V = V.toarray() if hasattr(V, "toarray") else V

# simple NaN fill
gene_means = np.nanmean(V, axis=0)
inds = np.where(np.isnan(V))
V[inds] = np.take(gene_means, inds[1])

V = StandardScaler(with_mean=False).fit_transform(V)

# -----------------------------
# FlowMap with precomputed embedding
# -----------------------------
emb = VectorFieldEmbedder(
    X,
    V,
    X_emb=X_emb,
    dist_method="euclidean",
    dof=70,
    n_spline_points=4000
)

emb.fit_embedding(seed=123)

# -----------------------------
# Plot
# -----------------------------
fig = flowmap.plot.plot_velocity_stream(
    X_2d=emb.X_emb,
    spline=emb.spline_vf,
    scatter_color=adata.obs["state_info"].values,
    cmap="tab10",
)

fig = flowmap.plot.plot_velocity_stream(
    X_2d=emb.X_emb,
    spline=emb.spline_vf,
    scatter_color=adata.obs["time_info"].astype(str).values,
    cmap="tab10",
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.neighbors import NearestNeighbors
from flowmap.utils import compute_velocity_on_grid


# ------------------------------------------------------------
# Helper: remove grid seeds outside the manifold
# ------------------------------------------------------------
def points_inside_mask(X_emb, seeds, k=8, radius_scale=1.2):
    nn = NearestNeighbors(n_neighbors=k).fit(X_emb)
    r = np.median(nn.kneighbors(X_emb)[0][:, -1]) * radius_scale
    neigh_idx = nn.radius_neighbors(seeds, radius=r, return_distance=False)
    return np.array([len(ix) > 0 for ix in neigh_idx])


# ------------------------------------------------------------
# Helper: manually add arrows
# ------------------------------------------------------------
def add_manual_arrows(ax, coords, X_emb, V_pred):
    nn = NearestNeighbors(n_neighbors=1).fit(X_emb)
    _, idx = nn.kneighbors(coords)
    idx = idx.ravel()

    ax.quiver(
        X_emb[idx,0], X_emb[idx,1],
        V_pred[idx,0], V_pred[idx,1],
        angles="xy",
        scale_units="xy",
        scale=3,
        width=0.003,
        headwidth=4.5,
        headlength=4.0,
        headaxislength=2.3,
        minlength=0.2,
        color="k",
        alpha=0.9,
    )


# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
X_emb = emb.X_emb
labels = np.asarray(adata.obs["state_info"].values)


# ------------------------------------------------------------
# Colors
# ------------------------------------------------------------
uniq = np.unique(labels)
other = [u for u in uniq if u != "Undifferentiated"]

cmap = plt.get_cmap("tab10", len(other))
colmap = {lab: mcolors.to_hex(cmap(i)) for i, lab in enumerate(other)}
colmap["Undifferentiated"] = "#d3d3d3"

cell_colors = np.array([colmap[l] for l in labels])


# ------------------------------------------------------------
# Velocity grid
# ------------------------------------------------------------
Xg, keep_mass, _ = compute_velocity_on_grid(X_emb, grid_size=25, min_mass=0.01)

keep_inside = points_inside_mask(X_emb, Xg)
Xg = Xg[keep_inside]

Vg = emb.spline_vf.predict(Xg)

def remove_grid_arrows(Xg, Vg, coords):
    """
    Remove arrows from the velocity grid by snapping to nearest grid seed.
    """
    if len(coords) == 0:
        return Xg, Vg

    nn = NearestNeighbors(n_neighbors=1).fit(Xg)
    _, idx = nn.kneighbors(coords)
    idx = np.unique(idx.ravel())

    keep = np.ones(len(Xg), dtype=bool)
    keep[idx] = False

    return Xg[keep], Vg[keep]

remove_coords = [
    [12.6,12.5]
]

Xg, Vg = remove_grid_arrows(Xg, Vg, remove_coords)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12,12))

# background cells
undiff = labels == "Undifferentiated"
ax.scatter(
    X_emb[undiff,0], X_emb[undiff,1],
    c="#d3d3d3",
    s=40,
    alpha=0.2,
    linewidths=0,
)

# foreground cells
mask = ~undiff
ax.scatter(
    X_emb[mask,0], X_emb[mask,1],
    c=cell_colors[mask],
    s=80,
    alpha=0.4,
    linewidths=0,
)

# velocity arrows
ax.quiver(
    Xg[:,0], Xg[:,1],
    Vg[:,0], Vg[:,1],
    angles="xy",
    scale_units="xy",
    scale=3,
    width=0.003,
    headwidth=4.5,
    headlength=4.0,
    headaxislength=2.3,
    minlength=0.2,
    color="k",
    alpha=0.9,
)


# ------------------------------------------------------------
# Manual arrow patches
# ------------------------------------------------------------
patch_coords = [
    [12.0,10.4],
    [14,3.6],
    [12.5,3.6],
    [12.6,12.5]
]

V_cells = emb.spline_vf.predict(X_emb)

add_manual_arrows(ax, patch_coords, X_emb, V_cells)


# ------------------------------------------------------------
# Styling
# ------------------------------------------------------------
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig(
    "./figures/larry/larry_embedding.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

X_emb = emb.X_emb
# Example patch coordinates (replace with your list)
patch_coords = [
    [12.0,10.4],
    [14,3.6],
    [12.5,3.6],
    [12.6,12.5]
]

fig, ax = plt.subplots(figsize=(10, 10))

# Scatter all cells
ax.scatter(
    X_emb[:, 0], X_emb[:, 1],
    c=cell_colors, s=30, alpha=0.4, linewidths=0
)

# Overlay patch coordinates
patch_coords = np.array(patch_coords)
ax.scatter(
    patch_coords[:, 0], patch_coords[:, 1],
    c="red", s=80, marker="x", label="Patch coords"
)

# Equal aspect
ax.set_aspect("equal")

# Dense grid
ax.grid(True, linestyle="--", alpha=0.6, color="grey")
ax.xaxis.set_major_locator(MultipleLocator(1))   # grid every 1 unit
ax.yaxis.set_major_locator(MultipleLocator(1))   # grid every 1 unit
ax.minorticks_on()                               # finer ticks
ax.xaxis.set_minor_locator(MultipleLocator(0.5))
ax.yaxis.set_minor_locator(MultipleLocator(0.5))
ax.grid(which="minor", linestyle=":", alpha=0.3)

# Legend
ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
plt.show()

In [ ]:
# === 2) Legend-only plot (includes Undifferentiated) ========================
fig, ax = plt.subplots(figsize=(4, 6))

handles = [plt.Line2D([], [], marker='o', linestyle='',
                      color=colmap[lab], label=lab, markersize=8)
           for lab in uniq]  # <-- now includes "Undifferentiated"

ax.legend(handles=handles, fontsize=10, loc='center')
ax.axis("off")

plt.tight_layout()
plt.savefig("./figures/larry/larry_embedding_legend.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import joblib

# Attach gene names (same ordering used in X)
emb.gene_names = adata.var_names[adata.var["highly_variable"]].values

# Save object
joblib.dump(emb, "./data/larry/larry_flowmap_embedder.pkl")

print("Saved FlowMap embedder with gene names 🎉")